In [ ]:
import os
import glob
import re
import csv

import cv2
import numpy as np
import torch

import time

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


raiz_entrada = r"datos"
raiz_salida = r"resultados"

ruta_checkpoint = r"vit_h.pth"
tipo_modelo = "vit_h"
dispositivo = "cpu"

reduccion_ancho_roi_predictor = 0.20
mitad_alto_roi_predictor = 140

excluir_inferior_amg = 0.25
dy_min_amg = 275
recorte_lateral_amg_ratio = 0.025

peso_brillo = 0.60
peso_hw = 0.40

umbral_area_aumenta = 1.75
umbral_area_reduce = 0.50

mitad_alto_roi_fallback_amg = 160

amg_points_per_side = 32
amg_pred_iou_thresh = 0.80
amg_stability_score_thresh = 0.85
amg_crop_n_layers = 1
amg_crop_overlap_ratio = 0.2
amg_min_mask_region_area = 200

carpetas_a_procesar = []  # vacio = todas; p.ej. ["T3_02"] para una sola

os.makedirs(raiz_salida, exist_ok=True)

def guardar_curva_distancia(filas, ruta_png, titulo):
    # Distancia P1-P2 (px) con eje derecho reescalado a deformacion (%). Colores aptos
    # para deuteranopia (azul / naranja), nunca verde ni amarillo.
    if not filas:
        return
    frames = [fila[0] for fila in filas]
    distancias = [fila[6] for fila in filas]
    d0 = distancias[0] if distancias[0] else 1.0

    fig, ax_izq = plt.subplots(figsize=(9, 5))
    ax_izq.plot(frames, distancias, color="#1f77b4", linewidth=2)
    ax_izq.set_xlabel("Frame")
    ax_izq.set_ylabel("Distancia P1-P2 (px)", color="#1f77b4")
    ax_izq.tick_params(axis="y", labelcolor="#1f77b4")
    ax_izq.grid(True, alpha=0.3)

    ax_der = ax_izq.twinx()
    lo, hi = ax_izq.get_ylim()
    ax_der.set_ylim((lo - d0) / d0 * 100.0, (hi - d0) / d0 * 100.0)
    ax_der.set_ylabel("Deformacion respecto al frame 0 (%)", color="#ff7f0e")
    ax_der.tick_params(axis="y", labelcolor="#ff7f0e")

    ax_izq.set_title(titulo)
    fig.tight_layout()
    fig.savefig(ruta_png, dpi=150, bbox_inches="tight")
    plt.close(fig)

In [ ]:
def clave_natural(path):
    nombre_archivo = os.path.basename(path)

    trozos = re.split(r"(\d+)", nombre_archivo)

    clave = []
    for t in trozos:
        if t.isdigit():
            clave.append(int(t))
        else:
            clave.append(t.lower())

    return clave


def a_uint8(imagen):
    if imagen.ndim == 3:
        canales = imagen.shape[2]
        if canales == 3:
            imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
        elif canales == 4:
            imagen = cv2.cvtColor(imagen, cv2.COLOR_BGRA2GRAY)

    if imagen.dtype == np.uint8:
        return imagen

    g = imagen.astype(np.float32)

    p1, p99 = np.percentile(g, [1, 99])
    if p99 <= p1:
        p1 = float(g.min())
        p99 = float(g.max())

    g = np.clip(g, p1, p99)

    g = g - p1
    denominador = (p99 - p1)
    if denominador > 0:
        g = g / denominador

    g = (g * 255.0).astype(np.uint8)
    return g


def gris_a_rgb(gris_uint8):
    canales = [gris_uint8, gris_uint8, gris_uint8]
    imagen_rgb = np.stack(canales, axis=-1)
    return imagen_rgb


def centroide_desde_mascara(mascara_bool):
    filas, columnas = np.where(mascara_bool)

    hay_pixeles = columnas.size > 0
    if not hay_pixeles:
        return None

    x_media = columnas.mean()
    y_media = filas.mean()

    centroide = np.array([x_media, y_media], dtype=np.float32)
    return centroide


def bbox_desde_mascara(mascara_bool):
    filas, columnas = np.where(mascara_bool)

    hay_pixeles = columnas.size > 0
    if not hay_pixeles:
        return None

    x0 = int(columnas.min())
    x1 = int(columnas.max())
    y0 = int(filas.min())
    y1 = int(filas.max())

    return x0, y0, x1, y1


def score_hw_desde_bbox(bbox_xyxy):
    x0, y0, x1, y1 = bbox_xyxy

    ancho = float((x1 - x0) + 1)
    alto = float((y1 - y0) + 1)

    if ancho <= 0 or alto <= 0:
        return 0.0

    ratio = alto / ancho

    score = ratio
    if ratio > 1.0:
        score = 1.0 / ratio

    return float(score)


def brillo_medio_mascara(gris_uint8_original, mascara_bool):
    valores = gris_uint8_original[mascara_bool]

    hay_pixeles = valores.size > 0
    if not hay_pixeles:
        return 0.0

    media = float(valores.mean())
    return media


def normalizar_brillos_por_frame(lista_brillos):
    if not lista_brillos:
        return []

    maximo = float(max(lista_brillos))
    if maximo <= 0:
        lista = []
        for _ in lista_brillos:
            lista.append(0.0)
        return lista

    lista = []
    for b in lista_brillos:
        lista.append(float(b) / maximo)

    return lista


def estimar_rango_x_probeta(gris_uint8, y0_ratio=0.10, y1_ratio=0.80, perc=35):
    alto_imagen, ancho_imagen = gris_uint8.shape

    y0 = int(y0_ratio * alto_imagen)
    y1 = int(y1_ratio * alto_imagen)

    zona = gris_uint8[y0:y1, :]
    media_por_columna = zona.mean(axis=0)

    umbral = np.percentile(media_por_columna, perc)
    mascara_oscuro = media_por_columna < umbral

    tramos = []
    en_tramo = False
    inicio_tramo = 0

    for i, es_oscuro in enumerate(mascara_oscuro):
        if es_oscuro and not en_tramo:
            en_tramo = True
            inicio_tramo = i

        if (not es_oscuro) and en_tramo:
            en_tramo = False
            fin_tramo = i - 1
            tramos.append((inicio_tramo, fin_tramo))

    if en_tramo:
        tramos.append((inicio_tramo, ancho_imagen - 1))

    if not tramos:
        return 0, ancho_imagen - 1

    centro_imagen_x = ancho_imagen * 0.5

    mejor_tramo = None
    mejor_valor = -1e18

    for x0, x1 in tramos:
        ancho_tramo = x1 - x0
        centro_tramo = (x0 + x1) * 0.5
        penalizacion_centro = 0.2 * abs(centro_tramo - centro_imagen_x)

        valor = ancho_tramo - penalizacion_centro
        if valor > mejor_valor:
            mejor_valor = valor
            mejor_tramo = (x0, x1)

    x0, x1 = mejor_tramo

    pad = int(0.03 * (x1 - x0 + 1))
    x0 = max(0, x0 - pad)
    x1 = min(ancho_imagen - 1, x1 + pad)

    return x0, x1


def recortar_roi_xyxy(cx, cy, ancho_roi, mitad_alto, ancho_imagen, alto_imagen):
    mitad_ancho = float(ancho_roi) * 0.5

    x0 = int(cx - mitad_ancho)
    x1 = int(cx + mitad_ancho)

    y0 = int(cy - mitad_alto)
    y1 = int(cy + mitad_alto)

    if x0 < 0:
        x0 = 0
    if x1 > (ancho_imagen - 1):
        x1 = ancho_imagen - 1

    if y0 < 0:
        y0 = 0
    if y1 > (alto_imagen - 1):
        y1 = alto_imagen - 1

    if x1 <= x0:
        x1 = min(ancho_imagen - 1, x0 + 1)

    if y1 <= y0:
        y1 = min(alto_imagen - 1, y0 + 1)

    caja_xyxy = np.array([x0, y0, x1, y1], dtype=np.float32)
    return caja_xyxy


def aplicar_mascara_en_roi(imagen_rgb, mascara_bool, caja_xyxy, color_rgb):
    x0, y0, x1, y1 = [int(v) for v in caja_xyxy.tolist()]

    roi_mascara = np.zeros(mascara_bool.shape, dtype=bool)
    roi_mascara[y0:y1 + 1, x0:x1 + 1] = True

    mascara_aplicar = mascara_bool & roi_mascara
    if not mascara_aplicar.any():
        return imagen_rgb

    salida = imagen_rgb.copy().astype(np.float32)

    alpha = 0.65
    color = np.array([color_rgb[0], color_rgb[1], color_rgb[2]], dtype=np.float32)

    pixeles = salida[mascara_aplicar]
    pixeles = (1.0 - alpha) * pixeles + alpha * color
    salida[mascara_aplicar] = pixeles

    salida = np.clip(salida, 0, 255).astype(np.uint8)
    return salida


def dibujar_caja_roi_blanca(imagen_rgb, caja_xyxy):
    x0, y0, x1, y1 = [int(v) for v in caja_xyxy.tolist()]
    salida = imagen_rgb.copy()
    cv2.rectangle(salida, (x0, y0), (x1, y1), (255, 255, 255), 2)
    return salida


def dibujar_centroide(imagen_rgb, centroide, texto):
    salida = imagen_rgb.copy()

    c = (int(centroide[0]), int(centroide[1]))
    cv2.circle(salida, c, 6, (255, 255, 0), 2)

    pos_texto = (c[0] + 8, c[1] - 8)
    cv2.putText(salida, texto, pos_texto, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    return salida

In [ ]:
def preprocesar_para_amg(gris_uint8_original, x0_probeta, x1_probeta, alto_imagen):
    ancho_probeta = float((x1_probeta - x0_probeta) + 1)
    margen = int(round(recorte_lateral_amg_ratio * ancho_probeta))

    x0_amg = int(x0_probeta + margen)
    x1_amg = int(x1_probeta - margen)

    if x1_amg <= x0_amg:
        x0_amg = int(x0_probeta)
        x1_amg = int(x1_probeta)

    alto_amg = int((1.0 - float(excluir_inferior_amg)) * alto_imagen)
    if alto_amg < 2:
        alto_amg = max(2, alto_imagen)

    return gris_uint8_original, x0_amg, x1_amg, alto_amg


def preprocesar_para_predictor(gris_uint8_original, lista_cajas_roi):
    gris_base = gris_uint8_original
    gris_retoque = gris_uint8_original
    return gris_base, gris_retoque

In [ ]:
def seleccionar_mejor_mascara_por_score(gris_uint8_original, lista_masks_bool):
    candidatos = []

    for mask_bool in lista_masks_bool:
        centroide = centroide_desde_mascara(mask_bool)
        if centroide is None:
            continue

        bbox = bbox_desde_mascara(mask_bool)
        if bbox is None:
            continue

        brillo = brillo_medio_mascara(gris_uint8_original, mask_bool)
        score_hw = score_hw_desde_bbox(bbox)

        candidatos.append(
            {
                "centroide": centroide,
                "segmentacion": mask_bool,
                "brillo": float(brillo),
                "score_hw": float(score_hw)
            }
        )

    if not candidatos:
        return None

    brillos = []
    for c in candidatos:
        brillos.append(c["brillo"])

    scores_brillo = normalizar_brillos_por_frame(brillos)

    for i in range(len(candidatos)):
        candidatos[i]["score_brillo"] = float(scores_brillo[i])

    for c in candidatos:
        score_total = (peso_brillo * c["score_brillo"]) + (peso_hw * c["score_hw"])
        c["score_total"] = float(score_total)

    def clave_por_score(c):
        return c["score_total"]

    candidatos.sort(key=clave_por_score, reverse=True)

    mejor = candidatos[0]
    return mejor["centroide"], mejor["segmentacion"]

def generar_mascaras_amg_en_recorte(generador_mascaras, imagen_rgb_recorte, x_offset, y_offset, alto_full, ancho_full):
    mascaras = generador_mascaras.generate(imagen_rgb_recorte)

    lista_full = []
    for m in mascaras:
        seg = m["segmentation"].astype(bool)

        alto_seg, ancho_seg = seg.shape

        seg_full = np.zeros((alto_full, ancho_full), dtype=bool)
        y1 = min(alto_full, y_offset + alto_seg)
        x1 = min(ancho_full, x_offset + ancho_seg)

        alto_pegar = y1 - y_offset
        ancho_pegar = x1 - x_offset

        if alto_pegar > 0 and ancho_pegar > 0:
            seg_full[y_offset:y1, x_offset:x1] = seg[:alto_pegar, :ancho_pegar]

        lista_full.append({"segmentation": seg_full})

    return lista_full


def seleccionar_dos_marcadores_amg_tiff00(gris_uint8_original, gris_uint8_amg, generador_mascaras, x0_amg, x1_amg, alto_amg):
    alto_imagen, ancho_imagen = gris_uint8_original.shape

    rgb_amg = gris_a_rgb(gris_uint8_amg)
    rgb_recorte = rgb_amg[:alto_amg, x0_amg:x1_amg + 1, :].copy()

    lista_mascaras_full = generar_mascaras_amg_en_recorte(
        generador_mascaras,
        rgb_recorte,
        x_offset=x0_amg,
        y_offset=0,
        alto_full=alto_imagen,
        ancho_full=ancho_imagen
    )

    candidatos = []
    for m in lista_mascaras_full:
        segmentacion = m["segmentation"].astype(bool)

        centroide = centroide_desde_mascara(segmentacion)
        if centroide is None:
            continue

        if not (x0_amg <= centroide[0] <= x1_amg):
            continue

        if not (centroide[1] <= float(alto_amg)):
            continue

        bbox = bbox_desde_mascara(segmentacion)
        if bbox is None:
            continue

        brillo = brillo_medio_mascara(gris_uint8_original, segmentacion)
        score_hw = score_hw_desde_bbox(bbox)

        candidatos.append(
            {
                "centroide": centroide,
                "segmentacion": segmentacion,
                "brillo": float(brillo),
                "score_hw": float(score_hw)
            }
        )

    if len(candidatos) < 2:
        return None

    brillos = []
    for c in candidatos:
        brillos.append(c["brillo"])

    scores_brillo = normalizar_brillos_por_frame(brillos)

    for i in range(len(candidatos)):
        candidatos[i]["score_brillo"] = float(scores_brillo[i])

    for c in candidatos:
        score_total = (peso_brillo * c["score_brillo"]) + (peso_hw * c["score_hw"])
        c["score_total"] = float(score_total)

    candidatos.sort(key=lambda d: d["score_total"], reverse=True)

    mejor_1 = candidatos[0]
    c1 = mejor_1["centroide"]
    m1 = mejor_1["segmentacion"]

    mejor_2 = None
    for c in candidatos[1:]:
        dy = abs(float(c["centroide"][1]) - float(c1[1]))
        if dy >= float(dy_min_amg):
            mejor_2 = c
            break

    if mejor_2 is None:
        return None

    c2 = mejor_2["centroide"]
    m2 = mejor_2["segmentacion"]

    if float(c2[1]) < float(c1[1]):
        c_tmp = c1
        m_tmp = m1
        c1 = c2
        m1 = m2
        c2 = c_tmp
        m2 = m_tmp

    return c1, c2, m1, m2


def segmentar_con_predictor(predictor_sam, imagen_rgb_predictor, gris_uint8_original, centroide_previo, caja_roi):
    predictor_sam.set_image(imagen_rgb_predictor)

    cx = float(centroide_previo[0])
    cy = float(centroide_previo[1])

    coords_punto = np.array([[cx, cy]], dtype=np.float32)
    etiquetas_punto = np.array([1], dtype=np.int32)

    mascaras, _, _ = predictor_sam.predict(
        point_coords=coords_punto,
        point_labels=etiquetas_punto,
        box=caja_roi,
        multimask_output=True
    )

    lista_masks = []
    numero = int(mascaras.shape[0])
    for i in range(numero):
        lista_masks.append(mascaras[i].astype(bool))

    seleccion = seleccionar_mejor_mascara_por_score(gris_uint8_original, lista_masks)
    if seleccion is None:
        mask_0 = lista_masks[0]
        centroide_0 = centroide_desde_mascara(mask_0)
        if centroide_0 is None:
            centroide_0 = centroide_previo.copy()
        return centroide_0, mask_0

    centroide, mask_bool = seleccion
    return centroide, mask_bool


def segmentar_con_amg_en_roi(generador_mascaras, gris_uint8_original, gris_uint8_retoque, caja_roi):
    alto_imagen, ancho_imagen = gris_uint8_original.shape

    x0, y0, x1, y1 = [int(v) for v in caja_roi.tolist()]
    gris_recorte = gris_uint8_retoque[y0:y1 + 1, x0:x1 + 1]
    if gris_recorte.size == 0:
        return None

    rgb_recorte = gris_a_rgb(gris_recorte)

    lista_mascaras_full = generar_mascaras_amg_en_recorte(
        generador_mascaras,
        rgb_recorte,
        x_offset=x0,
        y_offset=y0,
        alto_full=alto_imagen,
        ancho_full=ancho_imagen
    )

    lista_masks_bool = []
    for m in lista_mascaras_full:
        lista_masks_bool.append(m["segmentation"].astype(bool))

    seleccion = seleccionar_mejor_mascara_por_score(gris_uint8_original, lista_masks_bool)
    return seleccion


def crear_overlay_amg_tiff00(imagen_rgb_original, mascara_1, mascara_2, c1, c2):
    salida = imagen_rgb_original.copy()

    salida = aplicar_mascara_en_roi(salida, mascara_1, np.array([0, 0, imagen_rgb_original.shape[1] - 1, imagen_rgb_original.shape[0] - 1], dtype=np.float32), (0, 255, 0))
    salida = aplicar_mascara_en_roi(salida, mascara_2, np.array([0, 0, imagen_rgb_original.shape[1] - 1, imagen_rgb_original.shape[0] - 1], dtype=np.float32), (0, 0, 255))

    salida = dibujar_centroide(salida, c1, "M1")
    salida = dibujar_centroide(salida, c2, "M2")

    return salida


def crear_imagen_puntos_predictor(imagen_rgb_original, caja_1, mascara_1, centroide_1, caja_2, mascara_2, centroide_2):
    salida = imagen_rgb_original.copy()

    salida = dibujar_caja_roi_blanca(salida, caja_1)
    salida = dibujar_caja_roi_blanca(salida, caja_2)

    salida = aplicar_mascara_en_roi(salida, mascara_1, caja_1, (0, 255, 0))
    salida = aplicar_mascara_en_roi(salida, mascara_2, caja_2, (0, 0, 255))

    salida = dibujar_centroide(salida, centroide_1, "M1")
    salida = dibujar_centroide(salida, centroide_2, "M2")

    return salida

In [ ]:
modelo_sam = sam_model_registry[tipo_modelo](checkpoint=ruta_checkpoint)
modelo_sam.to(device=dispositivo)

predictor_sam = SamPredictor(modelo_sam)

generador_mascaras = SamAutomaticMaskGenerator(
    model=modelo_sam,
    points_per_side=amg_points_per_side,
    pred_iou_thresh=amg_pred_iou_thresh,
    stability_score_thresh=amg_stability_score_thresh,
    crop_n_layers=amg_crop_n_layers,
    crop_overlap_ratio=amg_crop_overlap_ratio,
    min_mask_region_area=amg_min_mask_region_area
)

print("SAM cargado en:", dispositivo)

In [ ]:
def procesar_carpeta(nombre_carpeta, predictor_sam, generador_mascaras):
    carpeta_imagenes = os.path.join(raiz_entrada, nombre_carpeta)

    nombre_carpeta_salida = f"{nombre_carpeta}_out"
    carpeta_salida = os.path.join(raiz_salida, nombre_carpeta_salida)
    os.makedirs(carpeta_salida, exist_ok=True)

    carpeta_puntos_predictor = os.path.join(carpeta_salida, "puntos predictor")
    os.makedirs(carpeta_puntos_predictor, exist_ok=True)

    patron_tif = os.path.join(carpeta_imagenes, "*.tif")
    patron_tiff = os.path.join(carpeta_imagenes, "*.tiff")

    rutas_tif = glob.glob(patron_tif)
    rutas_tiff = glob.glob(patron_tiff)

    rutas_frames = rutas_tif + rutas_tiff
    rutas_frames = sorted(rutas_frames, key=clave_natural)

    if not rutas_frames:
        return

    ruta_frame_0 = rutas_frames[0]
    imagen_raw_0 = cv2.imread(ruta_frame_0, cv2.IMREAD_UNCHANGED)

    gris_0_original = a_uint8(imagen_raw_0)
    alto_imagen, ancho_imagen = gris_0_original.shape

    x0_probeta, x1_probeta = estimar_rango_x_probeta(gris_0_original)
    ancho_probeta = float((x1_probeta - x0_probeta) + 1)
    ancho_roi_predictor = float(ancho_probeta) * (1.0 - float(reduccion_ancho_roi_predictor))

    gris_0_amg, x0_amg, x1_amg, alto_amg = preprocesar_para_amg(
        gris_0_original,
        x0_probeta,
        x1_probeta,
        alto_imagen
    )

    seleccion_0 = seleccionar_dos_marcadores_amg_tiff00(
        gris_0_original,
        gris_0_amg,
        generador_mascaras,
        x0_amg,
        x1_amg,
        alto_amg
    )

    if seleccion_0 is None:
        print(f"[{nombre_carpeta}] ERROR: no se pudieron seleccionar 2 marcadores en tiff00.")
        return

    c1_0, c2_0, m1_0, m2_0 = seleccion_0

    rgb_0_original = gris_a_rgb(gris_0_original)
    overlay_amg_0 = crear_overlay_amg_tiff00(rgb_0_original, m1_0, m2_0, c1_0, c2_0)
    overlay_amg_0_bgr = cv2.cvtColor(overlay_amg_0, cv2.COLOR_RGB2BGR)

    ruta_tiff00_amg = os.path.join(carpeta_salida, "tiff00_AMG_marcadores.png")
    cv2.imwrite(ruta_tiff00_amg, overlay_amg_0_bgr)

    filas_csv = []

    centroide_previo_1 = c1_0.copy()
    centroide_previo_2 = c2_0.copy()

    area_previa_1 = int(m1_0.sum())
    area_previa_2 = int(m2_0.sum())

    for indice_frame, ruta_frame in enumerate(rutas_frames):
        nombre_archivo = os.path.basename(ruta_frame)

        imagen_raw = cv2.imread(ruta_frame, cv2.IMREAD_UNCHANGED)

        gris_original = a_uint8(imagen_raw)
        alto_i, ancho_i = gris_original.shape

        caja_roi_1 = recortar_roi_xyxy(centroide_previo_1[0], centroide_previo_1[1], ancho_roi_predictor, mitad_alto_roi_predictor, ancho_i, alto_i)
        caja_roi_2 = recortar_roi_xyxy(centroide_previo_2[0], centroide_previo_2[1], ancho_roi_predictor, mitad_alto_roi_predictor, ancho_i, alto_i)

        lista_cajas = [caja_roi_1, caja_roi_2]

        gris_base, gris_retoque = preprocesar_para_predictor(gris_original, lista_cajas)

        rgb_predictor = gris_a_rgb(gris_retoque)

        if indice_frame == 0:
            c1 = c1_0.copy()
            c2 = c2_0.copy()
            m1 = m1_0
            m2 = m2_0
        else:
            c1_pred, m1_pred = segmentar_con_predictor(predictor_sam, rgb_predictor, gris_original, centroide_previo_1, caja_roi_1)
            area_1_pred = int(m1_pred.sum())

            fallo_1 = False
            if area_previa_1 > 0:
                if float(area_1_pred) > float(umbral_area_aumenta) * float(area_previa_1):
                    fallo_1 = True
                if float(area_1_pred) < float(umbral_area_reduce) * float(area_previa_1):
                    fallo_1 = True

            if fallo_1:
                caja_fallback_1 = recortar_roi_xyxy(centroide_previo_1[0], centroide_previo_1[1], ancho_roi_predictor, mitad_alto_roi_fallback_amg, ancho_i, alto_i)
                sel_1 = segmentar_con_amg_en_roi(generador_mascaras, gris_original, gris_retoque, caja_fallback_1)
                if sel_1 is not None:
                    c1_amg, m1_amg = sel_1
                    c1 = c1_amg
                    m1 = m1_amg
                else:
                    c1 = c1_pred
                    m1 = m1_pred
            else:
                c1 = c1_pred
                m1 = m1_pred

            c2_pred, m2_pred = segmentar_con_predictor(predictor_sam, rgb_predictor, gris_original, centroide_previo_2, caja_roi_2)
            area_2_pred = int(m2_pred.sum())

            fallo_2 = False
            if area_previa_2 > 0:
                if float(area_2_pred) > float(umbral_area_aumenta) * float(area_previa_2):
                    fallo_2 = True
                if float(area_2_pred) < float(umbral_area_reduce) * float(area_previa_2):
                    fallo_2 = True

            if fallo_2:
                caja_fallback_2 = recortar_roi_xyxy(centroide_previo_2[0], centroide_previo_2[1], ancho_roi_predictor, mitad_alto_roi_fallback_amg, ancho_i, alto_i)
                sel_2 = segmentar_con_amg_en_roi(generador_mascaras, gris_original, gris_retoque, caja_fallback_2)
                if sel_2 is not None:
                    c2_amg, m2_amg = sel_2
                    c2 = c2_amg
                    m2 = m2_amg
                else:
                    c2 = c2_pred
                    m2 = m2_pred
            else:
                c2 = c2_pred
                m2 = m2_pred

        area_1 = int(m1.sum())
        area_2 = int(m2.sum())

        dx = float(c2[0] - c1[0])
        dy = float(c2[1] - c1[1])
        distancia_px = float(np.hypot(dx, dy))

        fila = [
            int(indice_frame),
            str(nombre_archivo),
            float(c1[0]),
            float(c1[1]),
            float(c2[0]),
            float(c2[1]),
            float(distancia_px)
        ]
        filas_csv.append(fila)

        rgb_original = gris_a_rgb(gris_original)

        imagen_puntos = crear_imagen_puntos_predictor(rgb_original, caja_roi_1, m1, c1, caja_roi_2, m2, c2)
        imagen_puntos_bgr = cv2.cvtColor(imagen_puntos, cv2.COLOR_RGB2BGR)

        ruta_img_puntos = os.path.join(carpeta_puntos_predictor, f"puntos_{indice_frame:04d}.png")
        cv2.imwrite(ruta_img_puntos, imagen_puntos_bgr)

        centroide_previo_1 = c1
        centroide_previo_2 = c2

        area_previa_1 = area_1
        area_previa_2 = area_2

    ruta_csv = os.path.join(carpeta_salida, "resultados_centroides_distancia.csv")
    with open(ruta_csv, "w", newline="", encoding="utf-8") as archivo:
        escritor = csv.writer(archivo)
        cabecera = ["frame", "archivo", "x_marcador_1", "y_marcador_1", "x_marcador_2", "y_marcador_2", "distancia_pixeles"]
        escritor.writerow(cabecera)
        escritor.writerows(filas_csv)

    ruta_curva = os.path.join(carpeta_salida, "curva_distancia_V5.png")
    guardar_curva_distancia(filas_csv, ruta_curva, f"{nombre_carpeta} - distancia P1-P2")

    carpeta_puntos = os.path.join(carpeta_salida, "puntos_unidos")
    os.makedirs(carpeta_puntos, exist_ok=True)
    for i, ruta_frame in enumerate(rutas_frames):
        gris_i = a_uint8(cv2.imread(ruta_frame, cv2.IMREAD_UNCHANGED))
        lienzo = cv2.cvtColor(gris_i, cv2.COLOR_GRAY2BGR)
        punto1 = (int(round(filas_csv[i][2])), int(round(filas_csv[i][3])))
        punto2 = (int(round(filas_csv[i][4])), int(round(filas_csv[i][5])))
        cv2.line(lienzo, punto1, punto2, (255, 255, 255), 2)
        cv2.circle(lienzo, punto1, 10, (0, 128, 255), 2)
        cv2.circle(lienzo, punto2, 10, (0, 128, 255), 2)
        cv2.putText(lienzo, "P1", (punto1[0] + 14, punto1[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 128, 255), 2)
        cv2.putText(lienzo, "P2", (punto2[0] + 14, punto2[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 128, 255), 2)
        cv2.putText(lienzo, f"frame {int(filas_csv[i][0])}  dist = {filas_csv[i][6]:.1f} px", (20, 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2)
        cv2.imwrite(os.path.join(carpeta_puntos, f"puntos_{int(filas_csv[i][0]):04d}.png"), lienzo)


nombres_en_raiz = os.listdir(raiz_entrada)

carpetas = []
for nombre in nombres_en_raiz:
    ruta_posible = os.path.join(raiz_entrada, nombre)
    es_carpeta = os.path.isdir(ruta_posible)
    if es_carpeta:
        carpetas.append(nombre)

carpetas_ordenadas = sorted(carpetas, key=clave_natural)

for nombre_carpeta in carpetas_ordenadas:
    if carpetas_a_procesar and nombre_carpeta not in carpetas_a_procesar:
        continue
    try:
        t0 = time.perf_counter()

        procesar_carpeta(nombre_carpeta, predictor_sam, generador_mascaras)

        t1 = time.perf_counter()
        segundos = t1 - t0
        print(f"[{nombre_carpeta}] Tiempo: {segundos:.1f} s ({segundos/60:.2f} min)")

    except Exception as excepcion:
        print(f"[{nombre_carpeta}] EXCEPCION: {excepcion}")

print("FIN.")